In [ ]:
!pip install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 25.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  

In [ ]:
###############################################################################
# GPTQ BACKEND INSTALL — try multiple approaches, best to fallback
###############################################################################
import subprocess, sys, os

def try_install(label, cmd):
    print(f"\n{'─'*50}")
    print(f"Attempt: {label}")
    print(f"{'─'*50}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✔ Install succeeded")
        return True
    else:
        # Show last 3 lines of error only
        err_lines = [l for l in result.stderr.strip().split('\n') if l.strip()]
        for line in err_lines[-3:]:
            print(f"  {line.strip()}")
        print(f"  ✗ Failed")
        return False

def check_gptq():
    """Check if any GPTQ backend is importable."""
    for lib in ["gptqmodel", "auto_gptq"]:
        try:
            mod = __import__(lib)
            v = getattr(mod, '__version__', 'unknown')
            print(f"\n✅ {lib} {v} is available!")
            return lib
        except ImportError:
            continue
    return None

# Check if already installed
result = check_gptq()
if result:
    print("Nothing to do.")
else:
    attempts = [
        # 1. Pre-built wheels from HF index (fastest, best)
        ("auto-gptq from HF cu124 wheels",
         f"{sys.executable} -m pip install -q auto-gptq "
         f"--extra-index-url https://huggingface.github.io/autogptq-index/whl/cu124/"),

        # 2. gptqmodel from PyPI
        ("gptqmodel from PyPI",
         f"{sys.executable} -m pip install -q gptqmodel --no-build-isolation"),

        # 3. gptqmodel from GitHub source
        ("gptqmodel from GitHub",
         f"{sys.executable} -m pip install -q "
         f"git+https://github.com/ModelCloud/GPTQModel.git --no-build-isolation"),

        # 4. GUARANTEED FALLBACK: auto-gptq WITHOUT CUDA kernels
        #    Uses pure PyTorch — slower inference but correct results.
        #    Accuracy numbers (what your paper needs) are identical.
        ("auto-gptq WITHOUT CUDA ext (pure PyTorch fallback)",
         f"BUILD_CUDA_EXT=0 {sys.executable} -m pip install -q auto-gptq --no-build-isolation"),
    ]

    for label, cmd in attempts:
        if try_install(label, cmd):
            result = check_gptq()
            if result:
                break
    else:
        print("\n❌ All attempts failed. Check errors above.")

# Final verification — can transformers see the backend?
if check_gptq():
    print("\n🔍 Quick load test...")
    try:
        from transformers import AutoModelForCausalLM
        # Just check that the GPTQ config class is recognized
        from transformers import GPTQConfig
        print("✔ transformers GPTQ integration ready")
    except ImportError:
        print("⚠️  GPTQConfig not in this transformers version, but from_pretrained should still work")

    print("\n✅ GPTQ backend ready — proceed to Block 2")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.7.0
Torch        : 2.10.0+cu128
Triton       : 3.6.0



✅ gptqmodel 7.0.0 is available!
Nothing to do.

✅ gptqmodel 7.0.0 is available!

🔍 Quick load test...
✔ transformers GPTQ integration ready

✅ GPTQ backend ready — proceed to Block 2


In [ ]:
###############################################################################
# LLAMA 3.1 8B INSTRUCT — GSM8K QUANTIZATION ANALYSIS (BF16, INT8, NF4, GPTQ)
# BLOCK 1: Setup — Installs, imports, caching, stratified sampling, model check
# Single cell — copy-paste into Colab. GPU runtime required (L4 recommended).
# Drive must be mounted at /content/drive
# HuggingFace authentication must be active (huggingface-cli login)
###############################################################################

# ── 0. Installs ──────────────────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    for p in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", p],
                stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            print(f"⚠️  Could not install {p} — skipping")

def pip_install_special(pkg, *extra_args):
    """For packages needing special flags like --no-build-isolation."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg, *extra_args],
            stderr=subprocess.DEVNULL,
        )
    except subprocess.CalledProcessError:
        print(f"⚠️  Could not install {pkg} — skipping")

pip_install(
    "transformers>=4.43",
    "accelerate>=0.30",
    "bitsandbytes>=0.43",
    "optimum>=1.19",
    "datasets",
    "sentencepiece",
    "protobuf",
    "tqdm",
)

# GPTQModel (successor to auto-gptq) — needs torch visible at build time
# This also handles models originally quantized with AutoGPTQ
pip_install_special("gptqmodel", "--no-build-isolation")

# ── 1. Imports ───────────────────────────────────────────────────────────────
import os, re, gc, json, time, random
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ── 2. Paths & constants ────────────────────────────────────────────────────
SEED = 42
SAVE_DIR = "/content/drive/MyDrive/Papers/Wu"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_CACHE = "/content/model_cache"
os.makedirs(MODEL_CACHE, exist_ok=True)

DEVICE = "cuda"
assert torch.cuda.is_available(), "GPU required — switch runtime to GPU."

# ── SPEED KNOBS ──────────────────────────────────────────────────────────────
MAX_NEW_TOKENS     = 256        # most GSM8K answers finish <200 tok
SAMPLES_PER_BUCKET = 50         # 150 total (50 easy + 50 medium + 50 hard)
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning step by step."

BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
GPTQ_MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"

CONFIGS = [
    {"name": "BF16", "model_id": BASE_MODEL_ID, "quant": None},
    {"name": "INT8", "model_id": BASE_MODEL_ID, "quant": "int8"},
    {"name": "NF4",  "model_id": BASE_MODEL_ID, "quant": "nf4"},
    {"name": "GPTQ", "model_id": GPTQ_MODEL_ID, "quant": "gptq"},
]

# ── 3. Smart caching — reuse indices & skip finished configs ────────────────
# NOTE: We reuse the SAME eval_indices.json from the Mistral run so that
# both models are evaluated on the exact same 150 problems (paired comparison).
INDICES_PATH  = os.path.join(SAVE_DIR, "eval_indices.json")
COMBINED_PATH = os.path.join(SAVE_DIR, "llama_all_results.json")

def load_existing_results():
    """Load any per-config results already on Drive."""
    done = {}
    for cfg in CONFIGS:
        p = os.path.join(SAVE_DIR, f"llama_{cfg['name'].lower()}_results.json")
        if os.path.exists(p):
            try:
                with open(p) as f:
                    data = json.load(f)
                if data.get("summary", {}).get("status") == "COMPLETED":
                    done[cfg["name"]] = data
                    print(f"  ✔ Found completed results for {cfg['name']} — will skip")
            except Exception:
                pass
    return done

print("=" * 60)
print("CHECKING FOR CACHED RESULTS ON DRIVE")
print("=" * 60)
existing = load_existing_results()
if not existing:
    print("  No cached results found — will run all configs.")

# ── 4. Stratified sampling (load from cache or create) ──────────────────────
print("\n" + "=" * 60)
print("STEP 1: Stratified sampling")
print("=" * 60)

def count_steps(solution: str) -> int:
    sentences = re.split(r'(?<=[.!?])\s+', solution.strip())
    return max(1, len([s for s in sentences if len(s.strip()) > 5]))

def assign_bucket(n: int) -> str:
    if n <= 3:  return "easy"
    if n <= 5:  return "medium"
    return "hard"

if os.path.exists(INDICES_PATH):
    print(f"  Loading cached indices from {INDICES_PATH}")
    print(f"  (Same indices used by Mistral — paired comparison ✔)")
    with open(INDICES_PATH) as f:
        idx_data = json.load(f)
    sampled_indices = idx_data["indices"]
    bucket_labels   = {int(k): v for k, v in idx_data["bucket_labels"].items()}
    step_counts     = {int(k): v for k, v in idx_data["step_counts"].items()}
    print(f"  Loaded {len(sampled_indices)} indices "
          f"(Easy: {sum(1 for v in bucket_labels.values() if v=='easy')}, "
          f"Medium: {sum(1 for v in bucket_labels.values() if v=='medium')}, "
          f"Hard: {sum(1 for v in bucket_labels.values() if v=='hard')})")
else:
    print("  No cached indices — creating stratified sample...")
    dataset_full = load_dataset("openai/gsm8k", "main", split="test")
    print(f"  Full test set: {len(dataset_full)} problems")

    bucketed = {"easy": [], "medium": [], "hard": []}
    step_counts = {}
    for idx in range(len(dataset_full)):
        n = count_steps(dataset_full[idx]["answer"])
        step_counts[idx] = n
        bucketed[assign_bucket(n)].append(idx)

    print(f"  Pool — Easy: {len(bucketed['easy'])}, "
          f"Medium: {len(bucketed['medium'])}, Hard: {len(bucketed['hard'])}")

    random.seed(SEED)
    sampled_indices = []
    bucket_labels = {}
    for b in ["easy", "medium", "hard"]:
        chosen = random.sample(bucketed[b], min(SAMPLES_PER_BUCKET, len(bucketed[b])))
        for idx in chosen:
            bucket_labels[idx] = b
        sampled_indices.extend(chosen)

    with open(INDICES_PATH, "w") as f:
        json.dump({
            "seed": SEED,
            "samples_per_bucket": SAMPLES_PER_BUCKET,
            "indices": sampled_indices,
            "bucket_labels": {str(k): v for k, v in bucket_labels.items()},
            "step_counts":   {str(k): v for k, v in step_counts.items()},
        }, f, indent=2)
    print(f"  Saved {len(sampled_indices)} indices → {INDICES_PATH}")

# Load the actual dataset rows
dataset = load_dataset("openai/gsm8k", "main", split="test")
eval_subset = dataset.select(sampled_indices)
print(f"  Eval subset ready: {len(eval_subset)} problems")

# ── 5. Ensure models are downloaded (cache for reuse) ───────────────────────
print("\n" + "=" * 60)
print("STEP 2: Checking model availability")
print("=" * 60)

# Verify HuggingFace authentication (needed for gated Llama model)
try:
    from huggingface_hub import whoami
    user_info = whoami()
    print(f"  ✔ Authenticated as: {user_info.get('name', user_info.get('fullname', 'unknown'))}")
except Exception as e:
    print(f"  ⚠️  HuggingFace auth check failed: {e}")
    print("     Run: huggingface-cli login")
    print("     Make sure you have accepted the Llama 3.1 license at:")
    print("     https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct")

print(f"\n  Ensuring {BASE_MODEL_ID} tokenizer is cached...")
_ = AutoTokenizer.from_pretrained(BASE_MODEL_ID, cache_dir=MODEL_CACHE)
print("  ✔ Tokenizer cached")

needs_base = any(
    cfg["model_id"] == BASE_MODEL_ID and cfg["name"] not in existing
    for cfg in CONFIGS
)
if needs_base:
    print(f"  Pre-downloading base model weights (this only happens once)...")
    _m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        cache_dir=MODEL_CACHE,
        device_map="cpu",
    )
    del _m; gc.collect()
    print("  ✔ Base model weights cached")
else:
    print("  ✔ Base model not needed (all base configs done)")

needs_gptq = "GPTQ" not in existing
if needs_gptq:
    print(f"  Ensuring {GPTQ_MODEL_ID} is cached...")
    try:
        _ = AutoTokenizer.from_pretrained(GPTQ_MODEL_ID, cache_dir=MODEL_CACHE)
        print("  ✔ GPTQ tokenizer cached")
    except Exception as e:
        print(f"  ⚠️  GPTQ tokenizer issue: {e}")
        print("     Will attempt at runtime")

# ── 6. Environment info ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("ENVIRONMENT")
print(f"{'='*60}")
print(f"  torch:          {torch.__version__}")
print(f"  CUDA:           {torch.version.cuda}")
print(f"  GPU:            {torch.cuda.get_device_name(0)}")
print(f"  VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import importlib.metadata
for lib in ["transformers", "bitsandbytes", "optimum", "gptqmodel", "auto-gptq"]:
    try:
        v = importlib.metadata.version(lib)
        print(f"  {lib:<16} {v}")
    except:
        print(f"  {lib:<16} not installed")

print(f"\n{'='*60}")
print("SETUP COMPLETE")
print(f"{'='*60}")
print(f"  Model:          Llama 3.1 8B Instruct")
print(f"  Base ID:        {BASE_MODEL_ID}")
print(f"  GPTQ ID:        {GPTQ_MODEL_ID}")
print(f"  Configs:        {', '.join(c['name'] for c in CONFIGS)}")
print(f"  Eval problems:  {len(eval_subset)}")
print(f"  Already done:   {', '.join(existing.keys()) if existing else 'none'}")
print(f"  To run:         {', '.join(c['name'] for c in CONFIGS if c['name'] not in existing)}")
print(f"  Save dir:       {SAVE_DIR}")
print(f"\n  → Ready for Block 2 (inference loop)")

CHECKING FOR CACHED RESULTS ON DRIVE
  No cached results found — will run all configs.

STEP 1: Stratified sampling
  No cached indices — creating stratified sample...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

  ✔ GPTQ tokenizer cached

ENVIRONMENT
  torch:          2.10.0+cu128
  CUDA:           12.8
  GPU:            NVIDIA L4
  VRAM:           23.7 GB
  transformers     5.7.0
  bitsandbytes     0.49.2
  optimum          2.1.0
  gptqmodel        7.0.0
  auto-gptq        0.7.1

SETUP COMPLETE
  Model:          Llama 3.1 8B Instruct
  Base ID:        meta-llama/Meta-Llama-3.1-8B-Instruct
  GPTQ ID:        hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4
  Configs:        BF16, INT8, NF4, GPTQ
  Eval problems:  150
  Already done:   none
  To run:         BF16, INT8, NF4, GPTQ
  Save dir:       /content/drive/MyDrive/Papers/Wu

  → Ready for Block 2 (inference loop)


In [ ]:
###############################################################################
# LLAMA 3.1 8B INSTRUCT — GSM8K QUANTIZATION ANALYSIS (BF16, INT8, NF4, GPTQ)
# BLOCK 2 (CORRECTED): Answer extraction, model loading, inference loop, saving
# Paste into the NEXT Colab cell after Block 1 has run.
###############################################################################

import torch
import gc, json, time, re
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os, random, numpy as np
from datasets import load_dataset

# ── Block 1 defs (safe to re-run after kernel reset) ────────────────────────
SEED = 42
SAVE_DIR = "/content/drive/MyDrive/Papers/Wu"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_CACHE = "/content/model_cache"
os.makedirs(MODEL_CACHE, exist_ok=True)

DEVICE = "cuda"
assert torch.cuda.is_available(), "GPU required — switch runtime to GPU."

MAX_NEW_TOKENS     = 256
SAMPLES_PER_BUCKET = 50

SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning step by step."

BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
GPTQ_MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"

CONFIGS = [
    {"name": "BF16", "model_id": BASE_MODEL_ID, "quant": None},
    {"name": "INT8", "model_id": BASE_MODEL_ID, "quant": "int8"},
    {"name": "NF4",  "model_id": BASE_MODEL_ID, "quant": "nf4"},
    {"name": "GPTQ", "model_id": GPTQ_MODEL_ID, "quant": "gptq"},
]

INDICES_PATH  = os.path.join(SAVE_DIR, "eval_indices.json")
COMBINED_PATH = os.path.join(SAVE_DIR, "llama_all_results.json")

def load_existing_results():
    done = {}
    for cfg in CONFIGS:
        p = os.path.join(SAVE_DIR, f"llama_{cfg['name'].lower()}_results.json")
        if os.path.exists(p):
            try:
                with open(p) as f:
                    data = json.load(f)
                if data.get("summary", {}).get("status") == "COMPLETED":
                    done[cfg["name"]] = data
                    print(f"  ✔ Cached: {cfg['name']} — will skip")
            except Exception:
                pass
    return done

existing = load_existing_results()

def count_steps(solution: str) -> int:
    sentences = re.split(r'(?<=[.!?])\s+', solution.strip())
    return max(1, len([s for s in sentences if len(s.strip()) > 5]))

def assign_bucket(n: int) -> str:
    if n <= 3:  return "easy"
    if n <= 5:  return "medium"
    return "hard"

if os.path.exists(INDICES_PATH):
    with open(INDICES_PATH) as f:
        idx_data = json.load(f)
    sampled_indices = idx_data["indices"]
    bucket_labels   = {int(k): v for k, v in idx_data["bucket_labels"].items()}
    step_counts     = {int(k): v for k, v in idx_data["step_counts"].items()}
else:
    dataset_full = load_dataset("openai/gsm8k", "main", split="test")
    bucketed = {"easy": [], "medium": [], "hard": []}
    step_counts = {}
    for idx in range(len(dataset_full)):
        n = count_steps(dataset_full[idx]["answer"])
        step_counts[idx] = n
        bucketed[assign_bucket(n)].append(idx)
    random.seed(SEED)
    sampled_indices = []
    bucket_labels = {}
    for b in ["easy", "medium", "hard"]:
        chosen = random.sample(bucketed[b], min(SAMPLES_PER_BUCKET, len(bucketed[b])))
        for idx in chosen:
            bucket_labels[idx] = b
        sampled_indices.extend(chosen)
    with open(INDICES_PATH, "w") as f:
        json.dump({
            "seed": SEED, "samples_per_bucket": SAMPLES_PER_BUCKET,
            "indices": sampled_indices,
            "bucket_labels": {str(k): v for k, v in bucket_labels.items()},
            "step_counts":   {str(k): v for k, v in step_counts.items()},
        }, f, indent=2)

dataset = load_dataset("openai/gsm8k", "main", split="test")
eval_subset = dataset.select(sampled_indices)
print(f"Eval subset: {len(eval_subset)} problems")

# ── 6. Answer extraction ────────────────────────────────────────────────────

def normalize_number(x):
    if x is None: return None
    x = str(x).strip().replace(",", "")
    return x if x != "" else None

def extract_answer(text):
    if not text: return None
    m = re.search(r"####\s*(-?[0-9\.,]+)", text)
    if m: return normalize_number(m.group(1))
    matches = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return normalize_number(matches[-1]) if matches else None

def extract_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9\.,]+)", answer_text)
    if m: return normalize_number(m.group(1))
    return None

def numerically_equal(a, b, tol=1e-6):
    if a is None or b is None: return False
    try: return abs(float(a) - float(b)) <= tol
    except (ValueError, TypeError): return str(a).strip() == str(b).strip()

# ── 7. Model loading ────────────────────────────────────────────────────────

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    time.sleep(2)

def load_model_and_tokenizer(config):
    model_id = config["model_id"]
    quant    = config["quant"]

    # *** FIX: must use .from_pretrained(), not direct constructor ***
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, cache_dir=MODEL_CACHE, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    load_kwargs = dict(
        pretrained_model_name_or_path=model_id,
        cache_dir=MODEL_CACHE,
        device_map="auto",
        trust_remote_code=True,
    )

    if quant is None:                       # BF16
        load_kwargs["dtype"] = torch.bfloat16

    elif quant == "int8":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)

    elif quant == "nf4":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    elif quant == "gptq":
        pass  # pre-quantized checkpoint, auto-detected by transformers

    model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    model.eval()
    return model, tokenizer

# ── 8. Inference helper ─────────────────────────────────────────────────────

@torch.inference_mode()
def run_inference(model, tokenizer, question_text):
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + question_text},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}\n\n{question_text}"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs["input_ids"].to(model.device)
    attn_mask = inputs["attention_mask"].to(model.device)
    input_len = input_ids.shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids,
        attention_mask=attn_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    generated_ids = outputs[0, input_len:]
    output_text   = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return output_text, len(generated_ids), t1 - t0

# ── 9. Main experiment loop ─────────────────────────────────────────────────
all_per_question = []
all_summaries    = []

for cfg_name, data in existing.items():
    all_summaries.append(data["summary"])
    all_per_question.extend(data["per_question"])

configs_to_run = [c for c in CONFIGS if c["name"] not in existing]

if not configs_to_run:
    print("\n✅ All configs already completed! Loading cached results.")
else:
    print(f"\n{'='*60}")
    print(f"STEP 3: Running {len(configs_to_run)} configs "
          f"({', '.join(c['name'] for c in configs_to_run)})")
    print(f"{'='*60}")

for cfg_idx, config in enumerate(configs_to_run):
    print(f"\n{'='*60}")
    print(f"CONFIG {cfg_idx+1}/{len(configs_to_run)}: Llama-3.1-8B — {config['name']}")
    print(f"{'='*60}")

    free_gpu()

    try:
        print(f"  Loading model...")
        model, tokenizer = load_model_and_tokenizer(config)
        print(f"  ✔ Model loaded")
    except Exception as e:
        print(f"  ⚠️  FAILED to load {config['name']}: {e}")
        all_summaries.append({
            "model": "Llama-3.1-8B-Instruct",
            "quantization": config["name"],
            "status": "LOAD_FAILED",
            "error": str(e),
        })
        continue

    # Warmup (2 passes to let CUDA kernels compile)
    print("  Warmup pass...")
    try:
        _ = run_inference(model, tokenizer, "What is 2 + 2?")
        _ = run_inference(model, tokenizer, "What is 10 * 5?")
    except Exception as e:
        print(f"  Warmup issue (continuing): {e}")

    torch.cuda.reset_peak_memory_stats()

    correct_count = 0
    total_tokens  = 0
    total_latency = 0.0
    per_question_results = []

    n = len(eval_subset)
    pbar = tqdm(range(n), desc=f"  {config['name']}", unit="q",
                bar_format="  {l_bar}{bar:30}{r_bar}")

    for i in pbar:
        row          = eval_subset[i]
        original_idx = sampled_indices[i]
        gold_answer  = extract_gold_answer(row["answer"])
        bucket       = bucket_labels[original_idx]
        steps        = step_counts[original_idx]

        try:
            output_text, tokens_gen, latency = run_inference(
                model, tokenizer, row["question"]
            )
            predicted  = extract_answer(output_text)
            is_correct = numerically_equal(predicted, gold_answer)
        except Exception as e:
            output_text = f"ERROR: {e}"
            tokens_gen, latency, predicted, is_correct = 0, 0.0, None, False

        if is_correct: correct_count += 1
        total_tokens  += tokens_gen
        total_latency += latency

        record = {
            "model": "Llama-3.1-8B-Instruct",
            "quantization": config["name"],
            "prompt_type": "cot",
            "decoding": "greedy",
            "question_id": original_idx,
            "question_text": row["question"],
            "gold_answer": gold_answer,
            "prediction": predicted,
            "correct": is_correct,
            "tokens_generated": tokens_gen,
            "latency_sec": round(latency, 4),
            "output_text": output_text,
            "difficulty_bucket": bucket,
            "reasoning_steps": steps,
        }
        per_question_results.append(record)
        all_per_question.append(record)

        acc_so_far = correct_count / (i + 1) * 100
        pbar.set_postfix(acc=f"{acc_so_far:.1f}%", lat=f"{latency:.1f}s")

    pbar.close()

    # ── Summary ──────────────────────────────────────────────────────────
    peak_mem = torch.cuda.max_memory_allocated() / 1e6

    accuracy    = correct_count / n
    avg_latency = total_latency / n
    throughput  = total_tokens / total_latency if total_latency > 0 else 0

    bucket_acc = {}
    for b in ["easy", "medium", "hard"]:
        br = [r for r in per_question_results if r["difficulty_bucket"] == b]
        bc = sum(1 for r in br if r["correct"])
        bucket_acc[b] = {
            "correct": bc, "total": len(br),
            "accuracy": bc / len(br) if br else 0,
        }

    summary = {
        "model": "Llama-3.1-8B-Instruct",
        "quantization": config["name"],
        "model_id": config["model_id"],
        "prompt_type": "cot",
        "decoding": "greedy",
        "status": "COMPLETED",
        "accuracy": round(accuracy, 4),
        "accuracy_pct": round(accuracy * 100, 2),
        "correct": correct_count,
        "total": n,
        "avg_latency_sec": round(avg_latency, 4),
        "throughput_tokens_per_sec": round(throughput, 2),
        "avg_tokens_generated": round(total_tokens / n, 1),
        "peak_memory_mb": round(peak_mem, 1),
        "total_inference_time_min": round(total_latency / 60, 2),
        "bucket_accuracy": bucket_acc,
    }
    all_summaries.append(summary)

    print(f"\n  ── {config['name']} RESULTS ──")
    print(f"  Accuracy:     {accuracy*100:.1f}% ({correct_count}/{n})")
    print(f"  Easy:         {bucket_acc['easy']['accuracy']*100:.1f}%")
    print(f"  Medium:       {bucket_acc['medium']['accuracy']*100:.1f}%")
    print(f"  Hard:         {bucket_acc['hard']['accuracy']*100:.1f}%")
    print(f"  Avg latency:  {avg_latency:.2f}s")
    print(f"  Throughput:   {throughput:.1f} tok/s")
    print(f"  Peak memory:  {peak_mem:.0f} MB")

    cfg_path = os.path.join(SAVE_DIR, f"llama_{config['name'].lower()}_results.json")
    with open(cfg_path, "w") as f:
        json.dump({"summary": summary, "per_question": per_question_results},
                  f, indent=2, default=str)
    print(f"  Saved → {cfg_path}")

    del model, tokenizer
    free_gpu()
    print(f"  ✔ Model unloaded")

# ── 10. Save combined results ───────────────────────────────────────────────
print(f"\n{'='*60}")
print("SAVING COMBINED RESULTS")
print(f"{'='*60}")

combined = {
    "experiment": "Llama-3.1-8B-Instruct Quantization Analysis",
    "dataset": f"GSM8K (stratified {len(sampled_indices)}-problem sample)",
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "system_prompt": SYSTEM_PROMPT,
    "num_problems": len(sampled_indices),
    "sampling": {
        "method": "stratified",
        "per_bucket": SAMPLES_PER_BUCKET,
        "buckets": {
            "easy":   "2-3 reasoning steps",
            "medium": "4-5 reasoning steps",
            "hard":   "6+ reasoning steps",
        },
    },
    "configs_attempted": [c["name"] for c in CONFIGS],
    "summaries": all_summaries,
    "per_question_results": all_per_question,
}

with open(COMBINED_PATH, "w") as f:
    json.dump(combined, f, indent=2, default=str)
print(f"  Combined   → {COMBINED_PATH}")

summary_path = os.path.join(SAVE_DIR, "llama_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)
print(f"  Summary    → {summary_path}")

import transformers, bitsandbytes, importlib.metadata
env_info = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
}
try:
    env_info["gptqmodel"] = importlib.metadata.version("gptqmodel")
except: pass

env_path = os.path.join(SAVE_DIR, "llama_environment.json")
with open(env_path, "w") as f:
    json.dump(env_info, f, indent=2)
print(f"  Env info   → {env_path}")

# ── 11. Final table ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FINAL RESULTS — LLAMA 3.1 8B INSTRUCT")
print(f"{'='*60}")
print(f"{'Config':<8} {'Acc%':>6} {'Easy%':>6} {'Med%':>6} {'Hard%':>6} "
      f"{'Δ(H-E)':>7} {'Lat(s)':>7} {'Tok/s':>7} {'Mem(MB)':>8}")
print("-" * 72)

for s in all_summaries:
    if s.get("status") != "COMPLETED":
        print(f"{s['quantization']:<8} {'FAILED':>6}  — {s.get('error','')[:40]}")
        continue
    ba = s["bucket_accuracy"]
    delta = ba["hard"]["accuracy"] - ba["easy"]["accuracy"]
    print(f"{s['quantization']:<8} "
          f"{s['accuracy_pct']:>5.1f}% "
          f"{ba['easy']['accuracy']*100:>5.1f}% "
          f"{ba['medium']['accuracy']*100:>5.1f}% "
          f"{ba['hard']['accuracy']*100:>5.1f}% "
          f"{delta*100:>+6.1f}% "
          f"{s['avg_latency_sec']:>7.2f} "
          f"{s['throughput_tokens_per_sec']:>7.1f} "
          f"{s['peak_memory_mb']:>8.0f}")

print(f"\n✅ Done! All files in: {SAVE_DIR}")

Eval subset: 150 problems

STEP 3: Running 4 configs (BF16, INT8, NF4, GPTQ)

CONFIG 1/4: Llama-3.1-8B — BF16
  Loading model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ✔ Model loaded
  Warmup pass...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


    BF16:   0%|                              | 0/150 [00:00<?, ?q/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

INFO  Marlin fp16: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_extensions/marlin_fp16/293d6a0d3365ef60`.


INFO  Marlin fp16: torch.ops JIT extension ready in 197s (estimated ~117s, +80s).


INFO  gc.collect() reclaimed 0 objects in 0.284s                               


  ⚠️  FAILED to load GPTQ: type object 'BACKEND' has no attribute 'EXLLAMA_V1'

SAVING COMBINED RESULTS
  Combined   → /content/drive/MyDrive/Papers/Wu/llama_all_results.json
  Summary    → /content/drive/MyDrive/Papers/Wu/llama_summary.json
  Env info   → /content/drive/MyDrive/Papers/Wu/llama_environment.json

FINAL RESULTS — LLAMA 3.1 8B INSTRUCT
Config     Acc%  Easy%   Med%  Hard%  Δ(H-E)  Lat(s)   Tok/s  Mem(MB)
------------------------------------------------------------------------
BF16      55.3%  62.0%  68.0%  36.0%  -26.0%   14.32    15.8    16135
INT8      55.3%  60.0%  66.0%  40.0%  -20.0%   33.77     6.7     9169
NF4       62.7%  70.0%  66.0%  52.0%  -18.0%   15.51    14.1     5887
GPTQ     FAILED  — type object 'BACKEND' has no attribute '

✅ Done! All files in: /content/drive/MyDrive/Papers/Wu


In [ ]:
###############################################################################
# LLAMA 3.1 8B INSTRUCT — GPTQ ONLY (retry v2)
# Root cause: optimum/gptq/quantizer.py line 672 references BACKEND.EXLLAMA_V1
# which gptqmodel 7.0 removed from its enum. Fix: patch it back in.
###############################################################################

import torch, gc, json, time, re, os
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ── PATCH: Add missing EXLLAMA_V1 to BACKEND enum ──────────────────────────
# gptqmodel 7.0 removed EXLLAMA_V1/V2 from its BACKEND enum, but
# optimum's post_init_model still references it. We add it back as a
# dummy value so the comparison on line 672 simply evaluates to False
# and skips the exllama code path (Marlin is used instead).
try:
    from gptqmodel.utils.backend import BACKEND
    if not hasattr(BACKEND, 'EXLLAMA_V1'):
        # Add as a unique dummy value that won't match anything
        BACKEND.EXLLAMA_V1 = "EXLLAMA_V1_DUMMY"
        print("✔ Patched BACKEND.EXLLAMA_V1 (dummy)")
    if not hasattr(BACKEND, 'EXLLAMA_V2'):
        BACKEND.EXLLAMA_V2 = "EXLLAMA_V2_DUMMY"
        print("✔ Patched BACKEND.EXLLAMA_V2 (dummy)")
    if not hasattr(BACKEND, 'EXLLAMA'):
        BACKEND.EXLLAMA = "EXLLAMA_DUMMY"
        print("✔ Patched BACKEND.EXLLAMA (dummy)")
except ImportError:
    print("⚠️ Could not import BACKEND from gptqmodel — trying alternate path")
    try:
        from optimum.gptq.constants import BACKEND
        if not hasattr(BACKEND, 'EXLLAMA_V1'):
            BACKEND.EXLLAMA_V1 = "EXLLAMA_V1_DUMMY"
            print("✔ Patched via optimum path")
    except ImportError:
        print("⚠️ Could not patch — will attempt load anyway")

# ── Constants ───────────────────────────────────────────────────────────────
SEED = 42
SAVE_DIR = "/content/drive/MyDrive/Papers/Wu"
MODEL_CACHE = "/content/model_cache"
MAX_NEW_TOKENS = 256
SAMPLES_PER_BUCKET = 50
SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning step by step."
GPTQ_MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"
INDICES_PATH = os.path.join(SAVE_DIR, "eval_indices.json")

# ── Load indices & dataset ──────────────────────────────────────────────────
with open(INDICES_PATH) as f:
    idx_data = json.load(f)
sampled_indices = idx_data["indices"]
bucket_labels = {int(k): v for k, v in idx_data["bucket_labels"].items()}
step_counts = {int(k): v for k, v in idx_data["step_counts"].items()}

dataset = load_dataset("openai/gsm8k", "main", split="test")
eval_subset = dataset.select(sampled_indices)
print(f"Eval subset: {len(eval_subset)} problems")

# ── Helpers ─────────────────────────────────────────────────────────────────
def normalize_number(x):
    if x is None: return None
    x = str(x).strip().replace(",", "")
    return x if x != "" else None

def extract_answer(text):
    if not text: return None
    m = re.search(r"####\s*(-?[0-9\.,]+)", text)
    if m: return normalize_number(m.group(1))
    matches = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return normalize_number(matches[-1]) if matches else None

def extract_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9\.,]+)", answer_text)
    if m: return normalize_number(m.group(1))
    return None

def numerically_equal(a, b, tol=1e-6):
    if a is None or b is None: return False
    try: return abs(float(a) - float(b)) <= tol
    except (ValueError, TypeError): return str(a).strip() == str(b).strip()

# ── Free GPU ────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
time.sleep(2)

# ── Load model ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"GPTQ RETRY v2: {GPTQ_MODEL_ID}")
print(f"{'='*60}")

print("  Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    GPTQ_MODEL_ID, cache_dir=MODEL_CACHE, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("  Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    GPTQ_MODEL_ID,
    device_map="auto",
    cache_dir=MODEL_CACHE,
    trust_remote_code=True,
)
model.eval()
print(f"  ✔ Model loaded!")
print(f"  Peak VRAM: {torch.cuda.max_memory_allocated() / 1e6:.0f} MB")

# ── Inference function ──────────────────────────────────────────────────────
@torch.inference_mode()
def run_inference(model, tokenizer, question_text):
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + question_text},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}\n\n{question_text}"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs["input_ids"].to(model.device)
    attn_mask = inputs["attention_mask"].to(model.device)
    input_len = input_ids.shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids,
        attention_mask=attn_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    generated_ids = outputs[0, input_len:]
    output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return output_text, len(generated_ids), t1 - t0

# ── Warmup ──────────────────────────────────────────────────────────────────
print("  Warmup pass...")
_ = run_inference(model, tokenizer, "What is 2 + 2?")
_ = run_inference(model, tokenizer, "What is 10 * 5?")
print("  ✔ Warmup done")

# ── Inference loop ──────────────────────────────────────────────────────────
torch.cuda.reset_peak_memory_stats()

correct_count = 0
total_tokens = 0
total_latency = 0.0
per_question_results = []

n = len(eval_subset)
pbar = tqdm(range(n), desc="  GPTQ", unit="q",
            bar_format="  {l_bar}{bar:30}{r_bar}")

for i in pbar:
    row = eval_subset[i]
    original_idx = sampled_indices[i]
    gold_answer = extract_gold_answer(row["answer"])
    bucket = bucket_labels[original_idx]
    steps = step_counts[original_idx]

    try:
        output_text, tokens_gen, latency = run_inference(
            model, tokenizer, row["question"]
        )
        predicted = extract_answer(output_text)
        is_correct = numerically_equal(predicted, gold_answer)
    except Exception as e:
        output_text = f"ERROR: {e}"
        tokens_gen, latency, predicted, is_correct = 0, 0.0, None, False

    if is_correct: correct_count += 1
    total_tokens += tokens_gen
    total_latency += latency

    record = {
        "model": "Llama-3.1-8B-Instruct",
        "quantization": "GPTQ",
        "prompt_type": "cot",
        "decoding": "greedy",
        "question_id": original_idx,
        "question_text": row["question"],
        "gold_answer": gold_answer,
        "prediction": predicted,
        "correct": is_correct,
        "tokens_generated": tokens_gen,
        "latency_sec": round(latency, 4),
        "output_text": output_text,
        "difficulty_bucket": bucket,
        "reasoning_steps": steps,
    }
    per_question_results.append(record)

    acc_so_far = correct_count / (i + 1) * 100
    pbar.set_postfix(acc=f"{acc_so_far:.1f}%", lat=f"{latency:.1f}s")

pbar.close()

# ── Summary ─────────────────────────────────────────────────────────────────
peak_mem = torch.cuda.max_memory_allocated() / 1e6
accuracy = correct_count / n
avg_latency = total_latency / n
throughput = total_tokens / total_latency if total_latency > 0 else 0

bucket_acc = {}
for b in ["easy", "medium", "hard"]:
    br = [r for r in per_question_results if r["difficulty_bucket"] == b]
    bc = sum(1 for r in br if r["correct"])
    bucket_acc[b] = {
        "correct": bc, "total": len(br),
        "accuracy": bc / len(br) if br else 0,
    }

summary = {
    "model": "Llama-3.1-8B-Instruct",
    "quantization": "GPTQ",
    "model_id": GPTQ_MODEL_ID,
    "prompt_type": "cot",
    "decoding": "greedy",
    "status": "COMPLETED",
    "accuracy": round(accuracy, 4),
    "accuracy_pct": round(accuracy * 100, 2),
    "correct": correct_count,
    "total": n,
    "avg_latency_sec": round(avg_latency, 4),
    "throughput_tokens_per_sec": round(throughput, 2),
    "avg_tokens_generated": round(total_tokens / n, 1),
    "peak_memory_mb": round(peak_mem, 1),
    "total_inference_time_min": round(total_latency / 60, 2),
    "bucket_accuracy": bucket_acc,
}

print(f"\n  ── GPTQ RESULTS ──")
print(f"  Accuracy:     {accuracy*100:.1f}% ({correct_count}/{n})")
print(f"  Easy:         {bucket_acc['easy']['accuracy']*100:.1f}%")
print(f"  Medium:       {bucket_acc['medium']['accuracy']*100:.1f}%")
print(f"  Hard:         {bucket_acc['hard']['accuracy']*100:.1f}%")
print(f"  Avg latency:  {avg_latency:.2f}s")
print(f"  Throughput:   {throughput:.1f} tok/s")
print(f"  Peak memory:  {peak_mem:.0f} MB")

# ── Save ────────────────────────────────────────────────────────────────────
cfg_path = os.path.join(SAVE_DIR, "llama_gptq_results.json")
with open(cfg_path, "w") as f:
    json.dump({"summary": summary, "per_question": per_question_results},
              f, indent=2, default=str)
print(f"  Saved → {cfg_path}")

# ── Rebuild combined (all 4) ────────────────────────────────────────────────
print(f"\n{'='*60}")
print("REBUILDING COMBINED RESULTS")
print(f"{'='*60}")

all_summaries = []
all_per_question = []
for name in ["bf16", "int8", "nf4", "gptq"]:
    p = os.path.join(SAVE_DIR, f"llama_{name}_results.json")
    if os.path.exists(p):
        with open(p) as f:
            data = json.load(f)
        all_summaries.append(data["summary"])
        all_per_question.extend(data["per_question"])
        print(f"  ✔ llama_{name}_results.json")
    else:
        print(f"  ⚠️  Missing llama_{name}_results.json")

combined = {
    "experiment": "Llama-3.1-8B-Instruct Quantization Analysis",
    "dataset": f"GSM8K (stratified {len(sampled_indices)}-problem sample)",
    "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
    "system_prompt": SYSTEM_PROMPT,
    "num_problems": len(sampled_indices),
    "configs_attempted": ["BF16", "INT8", "NF4", "GPTQ"],
    "summaries": all_summaries,
    "per_question_results": all_per_question,
}

with open(os.path.join(SAVE_DIR, "llama_all_results.json"), "w") as f:
    json.dump(combined, f, indent=2, default=str)
with open(os.path.join(SAVE_DIR, "llama_summary.json"), "w") as f:
    json.dump(all_summaries, f, indent=2)

# ── Final table ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FINAL RESULTS — LLAMA 3.1 8B INSTRUCT (ALL 4)")
print(f"{'='*60}")
print(f"{'Config':<8} {'Acc%':>6} {'Easy%':>6} {'Med%':>6} {'Hard%':>6} "
      f"{'Δ(H-E)':>7} {'Lat(s)':>7} {'Tok/s':>7} {'Mem(MB)':>8}")
print("-" * 72)

for s in all_summaries:
    if s.get("status") != "COMPLETED":
        print(f"{s['quantization']:<8} {'FAILED':>6}")
        continue
    ba = s["bucket_accuracy"]
    delta = ba["hard"]["accuracy"] - ba["easy"]["accuracy"]
    print(f"{s['quantization']:<8} "
          f"{s['accuracy_pct']:>5.1f}% "
          f"{ba['easy']['accuracy']*100:>5.1f}% "
          f"{ba['medium']['accuracy']*100:>5.1f}% "
          f"{ba['hard']['accuracy']*100:>5.1f}% "
          f"{delta*100:>+6.1f}% "
          f"{s['avg_latency_sec']:>7.02f} "
          f"{s['throughput_tokens_per_sec']:>7.1f} "
          f"{s['peak_memory_mb']:>8.0f}")

print(f"\n✅ All 4 Llama configs done! Files in: {SAVE_DIR}")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

✔ Patched BACKEND.EXLLAMA_V1 (dummy)
✔ Patched BACKEND.EXLLAMA (dummy)
Eval subset: 150 problems

GPTQ RETRY v2: hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4
  Loading tokenizer...
  Loading model...


[transformers] `act_group_aware` has been auto-disabled as it is not compatible with `desc_act = True`.


INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                  


INFO  Kernel: selected -> `MarlinLinear`.                                      


Loading weights:   0%|          | 0/963 [00:00<?, ?it/s]

[transformers] LlamaForCausalLM LOAD REPORT from: hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
model.layers.{0...31}.self_attn.k_proj.bias | UNEXPECTED |  | 
model.layers.{0...31}.mlp.up_proj.bias      | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.v_proj.bias | UNEXPECTED |  | 
model.layers.{0...31}.mlp.down_proj.bias    | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.q_proj.bias | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.o_proj.bias | UNEXPECTED |  | 
model.layers.{0...31}.mlp.gate_proj.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO  gc.collect() reclaimed 120 objects in 0.294s                             


  ✔ Model loaded!
  Peak VRAM: 11481 MB
  Warmup pass...
  ✔ Warmup done


    GPTQ:   0%|                              | 0/150 [00:00<?, ?q/s]


  ── GPTQ RESULTS ──
  Accuracy:     56.0% (84/150)
  Easy:         68.0%
  Medium:       64.0%
  Hard:         36.0%
  Avg latency:  10.71s
  Throughput:   21.1 tok/s
  Peak memory:  11496 MB
  Saved → /content/drive/MyDrive/Papers/Wu/llama_gptq_results.json

REBUILDING COMBINED RESULTS
  ✔ llama_bf16_results.json
  ✔ llama_int8_results.json
  ✔ llama_nf4_results.json
  ✔ llama_gptq_results.json

FINAL RESULTS — LLAMA 3.1 8B INSTRUCT (ALL 4)
Config     Acc%  Easy%   Med%  Hard%  Δ(H-E)  Lat(s)   Tok/s  Mem(MB)
------------------------------------------------------------------------
BF16      55.3%  62.0%  68.0%  36.0%  -26.0%   14.32    15.8    16135
INT8      55.3%  60.0%  66.0%  40.0%  -20.0%   33.77     6.7     9169
NF4       62.7%  70.0%  66.0%  52.0%  -18.0%   15.51    14.1     5887
GPTQ      56.0%  68.0%  64.0%  36.0%  -32.0%   10.71    21.1    11496

✅ All 4 Llama configs done! Files in: /content/drive/MyDrive/Papers/Wu


In [ ]:
###############################################################################
# LLAMA 3.1 8B — REBUILD COMBINED RESULTS FROM PER-CONFIG FILES
# Safe to run anytime — reads whatever llama_*.json files exist on Drive
###############################################################################

import json, os

SAVE_DIR = "/content/drive/MyDrive/Papers/Wu"
SEED = 42
MAX_NEW_TOKENS = 256
SAMPLES_PER_BUCKET = 50
SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning step by step."

# ── Collect all per-config results ──────────────────────────────────────────
all_summaries = []
all_per_question = []
configs_found = []

for name in ["bf16", "int8", "nf4", "gptq"]:
    p = os.path.join(SAVE_DIR, f"llama_{name}_results.json")
    if os.path.exists(p):
        with open(p) as f:
            data = json.load(f)
        if data.get("summary", {}).get("status") == "COMPLETED":
            all_summaries.append(data["summary"])
            all_per_question.extend(data["per_question"])
            configs_found.append(name.upper())
            print(f"  ✔ llama_{name}_results.json — {data['summary']['accuracy_pct']}%")
        else:
            print(f"  ⚠️  llama_{name}_results.json exists but not COMPLETED")
    else:
        print(f"  ✗ llama_{name}_results.json — not found")

# ── Write combined JSON ─────────────────────────────────────────────────────
combined = {
    "experiment": "Llama-3.1-8B-Instruct Quantization Analysis",
    "dataset": "GSM8K (stratified 150-problem sample)",
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "system_prompt": SYSTEM_PROMPT,
    "num_problems": 150,
    "sampling": {
        "method": "stratified",
        "per_bucket": SAMPLES_PER_BUCKET,
        "buckets": {
            "easy": "2-3 reasoning steps",
            "medium": "4-5 reasoning steps",
            "hard": "6+ reasoning steps",
        },
    },
    "configs_attempted": ["BF16", "INT8", "NF4", "GPTQ"],
    "configs_completed": configs_found,
    "summaries": all_summaries,
    "per_question_results": all_per_question,
}

combined_path = os.path.join(SAVE_DIR, "llama_all_results.json")
with open(combined_path, "w") as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n  Combined → {combined_path}")

summary_path = os.path.join(SAVE_DIR, "llama_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)
print(f"  Summary  → {summary_path}")

# ── Final table ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("LLAMA 3.1 8B INSTRUCT — ALL RESULTS")
print(f"{'='*60}")
print(f"{'Config':<8} {'Acc%':>6} {'Easy%':>6} {'Med%':>6} {'Hard%':>6} "
      f"{'Δ(H-E)':>7} {'Lat(s)':>7} {'Tok/s':>7} {'Mem(MB)':>8}")
print("-" * 72)

for s in all_summaries:
    ba = s["bucket_accuracy"]
    delta = ba["hard"]["accuracy"] - ba["easy"]["accuracy"]
    print(f"{s['quantization']:<8} "
          f"{s['accuracy_pct']:>5.1f}% "
          f"{ba['easy']['accuracy']*100:>5.1f}% "
          f"{ba['medium']['accuracy']*100:>5.1f}% "
          f"{ba['hard']['accuracy']*100:>5.1f}% "
          f"{delta*100:>+6.1f}% "
          f"{s['avg_latency_sec']:>7.02f} "
          f"{s['throughput_tokens_per_sec']:>7.1f} "
          f"{s['peak_memory_mb']:>8.0f}")

print(f"\n✅ {len(configs_found)}/4 configs saved to: {SAVE_DIR}")

  ✔ llama_bf16_results.json — 55.33%
  ✔ llama_int8_results.json — 55.33%
  ✔ llama_nf4_results.json — 62.67%
  ✔ llama_gptq_results.json — 56.0%

  Combined → /content/drive/MyDrive/Papers/Wu/llama_all_results.json
  Summary  → /content/drive/MyDrive/Papers/Wu/llama_summary.json

LLAMA 3.1 8B INSTRUCT — ALL RESULTS
Config     Acc%  Easy%   Med%  Hard%  Δ(H-E)  Lat(s)   Tok/s  Mem(MB)
------------------------------------------------------------------------
BF16      55.3%  62.0%  68.0%  36.0%  -26.0%   14.32    15.8    16135
INT8      55.3%  60.0%  66.0%  40.0%  -20.0%   33.77     6.7     9169
NF4       62.7%  70.0%  66.0%  52.0%  -18.0%   15.51    14.1     5887
GPTQ      56.0%  68.0%  64.0%  36.0%  -32.0%   10.71    21.1    11496

✅ 4/4 configs saved to: /content/drive/MyDrive/Papers/Wu
